# Unit Tests Charts

Charts generated from `CI_TEST_RESULTS.md` section **1. Unit Tests** only.

- Source of truth: checked-in `test_report.md` counts listed in `CI_TEST_RESULTS.md`
- Includes only the 18 main comparison versions

## Setup

In [1]:
import os
import tempfile
from pathlib import Path

cache_dir = Path(tempfile.gettempdir()) / "matplotlib-cache"
cache_dir.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(cache_dir))

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

sns.set_theme(style="whitegrid", font_scale=1.25)
matplotlib.rcParams["figure.dpi"] = 150
matplotlib.rcParams["savefig.dpi"] = 300
matplotlib.rcParams["figure.facecolor"] = "white"
matplotlib.rcParams["axes.facecolor"] = "white"

REPORTS_DIR = Path("REPORTS")
if not REPORTS_DIR.exists():
    REPORTS_DIR = Path(".")

PASS_COLOR = "#2f9e44"
FAIL_COLOR = "#e03131"
RATE_COLOR = "#1c7ed6"
GRID_COLOR = "#dee2e6"

## Unit Test Data

In [2]:
unit_tests = pd.DataFrame([
    {"Version": "IMBP01", "Feature": "Inventory Management", "Strategy": "BP", "Tool": "Jest", "Passed": 7, "Failed": 0, "Total": 7},
    {"Version": "IMBP02", "Feature": "Inventory Management", "Strategy": "BP", "Tool": "Jest", "Passed": 5, "Failed": 2, "Total": 7},
    {"Version": "IMCE01", "Feature": "Inventory Management", "Strategy": "CE", "Tool": "Jest", "Passed": 7, "Failed": 0, "Total": 7},
    {"Version": "IMCE02", "Feature": "Inventory Management", "Strategy": "CE", "Tool": "Jest", "Passed": 7, "Failed": 0, "Total": 7},
    {"Version": "IMSD01", "Feature": "Inventory Management", "Strategy": "SD", "Tool": "vitest", "Passed": 7, "Failed": 0, "Total": 7},
    {"Version": "IMSD02", "Feature": "Inventory Management", "Strategy": "SD", "Tool": "vitest", "Passed": 7, "Failed": 0, "Total": 7},
    {"Version": "SCBP01", "Feature": "Shopping Cart", "Strategy": "BP", "Tool": "Jest", "Passed": 5, "Failed": 0, "Total": 5},
    {"Version": "SCBP02", "Feature": "Shopping Cart", "Strategy": "BP", "Tool": "Jest", "Passed": 4, "Failed": 1, "Total": 5},
    {"Version": "SCCE01", "Feature": "Shopping Cart", "Strategy": "CE", "Tool": "Jest", "Passed": 5, "Failed": 0, "Total": 5},
    {"Version": "SCCE02", "Feature": "Shopping Cart", "Strategy": "CE", "Tool": "node:test", "Passed": 0, "Failed": 5, "Total": 5},
    {"Version": "SCSD01", "Feature": "Shopping Cart", "Strategy": "SD", "Tool": "vitest", "Passed": 5, "Failed": 0, "Total": 5},
    {"Version": "SCSD02", "Feature": "Shopping Cart", "Strategy": "SD", "Tool": "vitest", "Passed": 5, "Failed": 0, "Total": 5},
    {"Version": "PDBP01", "Feature": "Promotions & Discounts", "Strategy": "BP", "Tool": "Jest", "Passed": 6, "Failed": 0, "Total": 6},
    {"Version": "PDBP02", "Feature": "Promotions & Discounts", "Strategy": "BP", "Tool": "Jest", "Passed": 1, "Failed": 5, "Total": 6},
    {"Version": "PDCE01", "Feature": "Promotions & Discounts", "Strategy": "CE", "Tool": "Jest", "Passed": 6, "Failed": 0, "Total": 6},
    {"Version": "PDCE02", "Feature": "Promotions & Discounts", "Strategy": "CE", "Tool": "node:test", "Passed": 5, "Failed": 1, "Total": 6},
    {"Version": "PDSD01", "Feature": "Promotions & Discounts", "Strategy": "SD", "Tool": "vitest", "Passed": 6, "Failed": 0, "Total": 6},
    {"Version": "PDSD02", "Feature": "Promotions & Discounts", "Strategy": "SD", "Tool": "Jest", "Passed": 5, "Failed": 1, "Total": 6},
])

feature_order = ["Inventory Management", "Shopping Cart", "Promotions & Discounts"]
strategy_order = ["BP", "CE", "SD"]
strategy_names = {"BP": "Basic Prompting", "CE": "Context Engineering", "SD": "Spec-Driven Development"}
feature_short_names = {
    "Inventory Management": "Inventory Mgmt",
    "Shopping Cart": "Shopping Cart",
    "Promotions & Discounts": "Promos & Discounts",
}
version_order = [
    "IMBP01", "IMBP02", "SCBP01", "SCBP02", "PDBP01", "PDBP02",
    "IMCE01", "IMCE02", "SCCE01", "SCCE02", "PDCE01", "PDCE02",
    "IMSD01", "IMSD02", "SCSD01", "SCSD02", "PDSD01", "PDSD02",
]

def contiguous_spans(values):
    spans = []
    start = 0
    for idx in range(1, len(values) + 1):
        if idx == len(values) or values[idx] != values[start]:
            spans.append((start, idx, values[start]))
            start = idx
    return spans

def draw_methodology_groups(ax, meta, y=-0.34, draw_feature_dividers=True):
    strategy_spans = contiguous_spans(meta["Strategy"].tolist())
    feature_spans = contiguous_spans(list(zip(meta["Strategy"], meta["Feature"])))
    strategy_boundaries = {end for _, end, _ in strategy_spans[:-1]}

    if draw_feature_dividers:
        for start, _, _ in feature_spans[1:]:
            if start not in strategy_boundaries:
                ax.axvline(start - 0.5, color="#ced4da", linewidth=1, linestyle="--", alpha=0.9)

    for start, _, _ in strategy_spans[1:]:
        ax.axvline(start - 0.5, color="#495057", linewidth=1.6, linestyle="-", alpha=0.7)

    for start, end, strategy in strategy_spans:
        ax.text((start + end - 1) / 2, y, strategy_names[strategy], ha="center", va="top",
                fontsize=10, fontweight="bold", transform=ax.get_xaxis_transform(), clip_on=False)

unit_tests["Pass Rate"] = unit_tests["Passed"] / unit_tests["Total"] * 100
unit_tests["Feature"] = pd.Categorical(unit_tests["Feature"], categories=feature_order, ordered=True)
unit_tests["Strategy"] = pd.Categorical(unit_tests["Strategy"], categories=strategy_order, ordered=True)
unit_tests["Version"] = pd.Categorical(unit_tests["Version"], categories=version_order, ordered=True)
unit_tests = unit_tests.sort_values("Version").reset_index(drop=True)

unit_tests

,Version,Feature,Strategy,Tool,Passed,Failed,Total,Pass Rate
0,IMBP01,Inventory Management,BP,Jest,7,0,7,100.000000
1,IMBP02,Inventory Management,BP,Jest,5,2,7,71.428571
2,SCBP01,Shopping Cart,BP,Jest,5,0,5,100.000000
3,SCBP02,Shopping Cart,BP,Jest,4,1,5,80.000000
4,PDBP01,Promotions & Discounts,BP,Jest,6,0,6,100.000000
5,PDBP02,Promotions & Discounts,BP,Jest,1,5,6,16.666667
6,IMCE01,Inventory Management,CE,Jest,7,0,7,100.000000
7,IMCE02,Inventory Management,CE,Jest,7,0,7,100.000000
8,SCCE01,Shopping Cart,CE,Jest,5,0,5,100.000000
9,SCCE02,Shopping Cart,CE,node:test,0,5,5,0.000000


## Strategy Summary

Strategy summary for the 18 main comparison versions.

In [3]:
strategy_source = unit_tests.copy()

strategy_summary = (
    strategy_source.groupby("Strategy", observed=False)[["Passed", "Failed", "Total"]]
    .sum()
    .reindex(strategy_order)
    .reset_index()
)
strategy_summary["Pass Rate"] = strategy_summary["Passed"] / strategy_summary["Total"] * 100
strategy_summary["Pass Rate Label"] = strategy_summary["Pass Rate"].round().astype(int).astype(str) + "%"

expected = pd.DataFrame([
    {"Strategy": "BP", "Passed": 28, "Failed": 8, "Total": 36, "Pass Rate Label": "78%"},
    {"Strategy": "CE", "Passed": 30, "Failed": 6, "Total": 36, "Pass Rate Label": "83%"},
    {"Strategy": "SD", "Passed": 35, "Failed": 1, "Total": 36, "Pass Rate Label": "97%"},
])

pd.testing.assert_frame_equal(
    strategy_summary[["Strategy", "Passed", "Failed", "Total", "Pass Rate Label"]].reset_index(drop=True),
    expected,
    check_dtype=False,
)

strategy_summary

,Strategy,Passed,Failed,Total,Pass Rate,Pass Rate Label
0,BP,28,8,36,77.777778,78%
1,CE,30,6,36,83.333333,83%
2,SD,35,1,36,97.222222,97%


---
## Chart 1: Unit Test Results by Version

Stacked pass/fail counts for every version listed in the Unit Tests section.

In [4]:
fig, ax = plt.subplots(figsize=(13.5, 6.2))

x = np.arange(len(unit_tests))
ax.bar(x, unit_tests["Passed"], color=PASS_COLOR, label="Passed")
ax.bar(x, unit_tests["Failed"], bottom=unit_tests["Passed"], color=FAIL_COLOR, label="Failed")

for idx, row in unit_tests.iterrows():
    ax.text(idx, row["Total"] + 0.35, f"{int(row['Passed'])}/{int(row['Total'])}", ha="center", va="bottom", fontsize=8.5)

ax.set_title("Unit Test Results by Version", fontsize=18, fontweight="bold", pad=14)
ax.set_ylabel("Number of Tests")
ax.set_xticks(x)
ax.set_xticklabels(unit_tests["Version"].astype(str), rotation=45, ha="right")
ax.set_ylim(0, unit_tests["Total"].max() + 3)
ax.grid(axis="y", color=GRID_COLOR)
ax.legend(loc="upper left", ncol=2, frameon=True)

draw_methodology_groups(ax, unit_tests, y=-0.34)

fig.subplots_adjust(bottom=0.29)
fig.savefig(REPORTS_DIR / "chart_unit_tests_by_version.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 2025x930 with 1 Axes>

---
## Chart 2: Unit Test Pass Rate by Version

Version-level pass rate, highlighting partial and failing implementations.

In [5]:
fig, ax = plt.subplots(figsize=(13.5, 5.6))

colors = [PASS_COLOR if rate == 100 else ("#f08c00" if rate >= 70 else FAIL_COLOR) for rate in unit_tests["Pass Rate"]]
bars = ax.bar(x, unit_tests["Pass Rate"], color=colors)

for bar, rate in zip(bars, unit_tests["Pass Rate"]):
    ax.text(bar.get_x() + bar.get_width() / 2, rate + 2, f"{rate:.0f}%", ha="center", va="bottom", fontsize=8.5)

ax.axhline(100, color="#495057", linewidth=1, linestyle=":")
ax.set_title("Unit Test Pass Rate by Version", fontsize=18, fontweight="bold", pad=14)
ax.set_ylabel("Pass Rate (%)")
ax.set_xticks(x)
ax.set_xticklabels(unit_tests["Version"].astype(str), rotation=45, ha="right")
ax.set_ylim(0, 112)
ax.grid(axis="y", color=GRID_COLOR)
draw_methodology_groups(ax, unit_tests, y=-0.37)

fig.subplots_adjust(bottom=0.32)
fig.savefig(REPORTS_DIR / "chart_unit_test_pass_rate_by_version.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 2025x840 with 1 Axes>

---
## Chart 3: Test Summary by Strategy

This chart mirrors `### Test Summary by Strategy` for Unit Tests only.

In [6]:
fig, ax = plt.subplots(figsize=(7.8, 5.8))

sx = np.arange(len(strategy_summary))
ax.bar(sx, strategy_summary["Passed"], color=PASS_COLOR, label="Passed")
ax.bar(sx, strategy_summary["Failed"], bottom=strategy_summary["Passed"], color=FAIL_COLOR, label="Failed")

for idx, row in strategy_summary.iterrows():
    ax.text(idx, row["Total"] + 1.2, f"{int(row['Passed'])}/{int(row['Total'])}\n{row['Pass Rate Label']}", ha="center", va="bottom", fontsize=11, fontweight="bold")

ax.set_title("Unit Test Summary by Strategy", fontsize=18, fontweight="bold", pad=14)
ax.set_ylabel("Number of Tests")
ax.set_xticks(sx)
ax.set_xticklabels(["BP\nBasic Prompting", "CE\nContext Engineering", "SD\nSpec-Driven Development"])
ax.set_ylim(0, strategy_summary["Total"].max() + 9)
ax.grid(axis="y", color=GRID_COLOR)
ax.legend(loc="upper left", ncol=2, frameon=True)

fig.tight_layout()
fig.savefig(REPORTS_DIR / "chart_unit_test_summary_by_strategy.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 1170x870 with 1 Axes>

---
## Chart 4: Pass Rate Heatmap by Feature and Strategy

Average pass rate by feature and strategy.

In [7]:
heatmap_data = (
    strategy_source.groupby(["Feature", "Strategy"], observed=False)[["Passed", "Total"]]
    .sum()
    .assign(**{"Pass Rate": lambda df: df["Passed"] / df["Total"] * 100})
    .reset_index()
    .pivot(index="Feature", columns="Strategy", values="Pass Rate")
    .reindex(index=feature_order, columns=strategy_order)
)

fig, ax = plt.subplots(figsize=(7.8, 4.8))
sns.heatmap(
    heatmap_data,
    annot=True,
    fmt=".0f",
    cmap="RdYlGn",
    vmin=0,
    vmax=100,
    linewidths=1,
    linecolor="white",
    cbar_kws={"label": "Pass Rate (%)"},
    ax=ax,
)
ax.set_title("Unit Test Pass Rate by Feature and Strategy", fontsize=16, fontweight="bold", pad=12)
ax.set_xlabel("Strategy")
ax.set_ylabel("Feature")

fig.tight_layout()
fig.savefig(REPORTS_DIR / "chart_unit_test_pass_rate_heatmap.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 1170x720 with 2 Axes>

---
## SonarQube Data

Source: live SonarCloud API (`reliability_issues`, `security_issues`, `maintainability_issues`, `duplicated_lines_density`, `security_hotspots`) — re-fetched 2026-06-06 after run #27056984423.

In [8]:
STRATEGY_COLORS = {"BP": "#e74c3c", "CE": "#3498db", "SD": "#2ecc71"}
STRATEGY_NAMES = strategy_names

sonar = pd.DataFrame([
    {"Version": "IMBP01", "Feature": "Inventory Management",   "Strategy": "BP", "Security": 3,  "Reliability": 3,  "Maintainability": 7,  "Duplications": 6.1,  "Hotspots": 0},
    {"Version": "IMBP02", "Feature": "Inventory Management",   "Strategy": "BP", "Security": 2,  "Reliability": 1,  "Maintainability": 1,  "Duplications": 0.0,  "Hotspots": 0},
    {"Version": "IMCE01", "Feature": "Inventory Management",   "Strategy": "CE", "Security": 7,  "Reliability": 19, "Maintainability": 27, "Duplications": 0.0,  "Hotspots": 6},
    {"Version": "IMCE02", "Feature": "Inventory Management",   "Strategy": "CE", "Security": 16, "Reliability": 2,  "Maintainability": 1,  "Duplications": 5.4,  "Hotspots": 4},
    {"Version": "IMSD01", "Feature": "Inventory Management",   "Strategy": "SD", "Security": 7,  "Reliability": 12, "Maintainability": 15, "Duplications": 2.7,  "Hotspots": 2},
    {"Version": "IMSD02", "Feature": "Inventory Management",   "Strategy": "SD", "Security": 5,  "Reliability": 0,  "Maintainability": 6,  "Duplications": 13.0, "Hotspots": 2},
    {"Version": "SCBP01", "Feature": "Shopping Cart",          "Strategy": "BP", "Security": 9,  "Reliability": 9,  "Maintainability": 20, "Duplications": 4.7,  "Hotspots": 2},
    {"Version": "SCBP02", "Feature": "Shopping Cart",          "Strategy": "BP", "Security": 6,  "Reliability": 2,  "Maintainability": 2,  "Duplications": 0.0,  "Hotspots": 1},
    {"Version": "SCCE01", "Feature": "Shopping Cart",          "Strategy": "CE", "Security": 8,  "Reliability": 9,  "Maintainability": 11, "Duplications": 3.8,  "Hotspots": 2},
    {"Version": "SCCE02", "Feature": "Shopping Cart",          "Strategy": "CE", "Security": 9,  "Reliability": 2,  "Maintainability": 1,  "Duplications": 0.0,  "Hotspots": 4},
    {"Version": "SCSD01", "Feature": "Shopping Cart",          "Strategy": "SD", "Security": 7,  "Reliability": 0,  "Maintainability": 5,  "Duplications": 21.9, "Hotspots": 3},
    {"Version": "SCSD02", "Feature": "Shopping Cart",          "Strategy": "SD", "Security": 1,  "Reliability": 3,  "Maintainability": 19, "Duplications": 0.0,  "Hotspots": 1},
    {"Version": "PDBP01", "Feature": "Promotions & Discounts", "Strategy": "BP", "Security": 8,  "Reliability": 1,  "Maintainability": 9,  "Duplications": 1.3,  "Hotspots": 3},
    {"Version": "PDBP02", "Feature": "Promotions & Discounts", "Strategy": "BP", "Security": 6,  "Reliability": 3,  "Maintainability": 2,  "Duplications": 0.0,  "Hotspots": 2},
    {"Version": "PDCE01", "Feature": "Promotions & Discounts", "Strategy": "CE", "Security": 6,  "Reliability": 16, "Maintainability": 20, "Duplications": 0.0,  "Hotspots": 4},
    {"Version": "PDCE02", "Feature": "Promotions & Discounts", "Strategy": "CE", "Security": 9,  "Reliability": 4,  "Maintainability": 8,  "Duplications": 0.0,  "Hotspots": 4},
    {"Version": "PDSD01", "Feature": "Promotions & Discounts", "Strategy": "SD", "Security": 5,  "Reliability": 6,  "Maintainability": 9,  "Duplications": 0.0,  "Hotspots": 5},
    {"Version": "PDSD02", "Feature": "Promotions & Discounts", "Strategy": "SD", "Security": 1,  "Reliability": 0,  "Maintainability": 13, "Duplications": 0.0,  "Hotspots": 0},
])

sonar["Strategy"] = pd.Categorical(sonar["Strategy"], categories=strategy_order, ordered=True)
sonar["Version"]  = pd.Categorical(sonar["Version"],  categories=version_order, ordered=True)
sonar = sonar.sort_values("Version").reset_index(drop=True)

# Strategy-level averages (verify against CI_TEST_RESULTS.md)
sonar_strategy = sonar.groupby("Strategy", observed=False)[["Security","Reliability","Maintainability","Duplications","Hotspots"]].mean().round(2)
sonar_strategy

,Security,Reliability,Maintainability,Duplications,Hotspots
Strategy,,,,,
BP,5.67,3.17,6.83,2.02,1.33
CE,9.17,8.67,11.33,1.53,4.00
SD,4.33,3.50,11.17,6.27,2.17


---
## Chart 5: SonarQube Metrics by Strategy

Average SonarQube metrics per strategy — lower is better. Security open issues and security hotspots are shown separately.

In [9]:
fig, axes = plt.subplots(2, 3, figsize=(14.4, 8.0))
strategies = ["BP", "CE", "SD"]
colors = [STRATEGY_COLORS[s] for s in strategies]
xlabels = strategies
sx = np.arange(len(strategies))

metrics = [
    ("Security",        "Security (Open)",         "Avg Security Open Issues",   axes[0][0]),
    ("Hotspots",        "Security Hotspots",       "Avg Security Hotspots",      axes[0][1]),
    ("Reliability",     "Reliability (Open)",      "Avg Reliability Open Issues", axes[0][2]),
    ("Maintainability", "Maintainability (Open)",  "Avg Maintainability Open Issues", axes[1][0]),
    ("Duplications",    "Duplications (%)",        "Avg Duplication %",          axes[1][1]),
]

for metric, title, ylabel, ax in metrics:
    vals = sonar_strategy[metric]
    bars = ax.bar(sx, vals, color=colors, edgecolor="white", width=0.38)
    for bar, val in zip(bars, vals):
        fmt = f"{val:.2f}%" if metric == "Duplications" else f"{val:.2f}"
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + max(vals) * 0.03,
                fmt, ha="center", va="bottom", fontweight="bold", fontsize=12)
    ax.set_xticks(sx)
    ax.set_xticklabels(xlabels, fontsize=10)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=8)
    ax.set_ylim(0, max(vals) * 1.35 + 0.5)
    ax.grid(axis="y", color=GRID_COLOR)

from matplotlib.patches import Patch

axes[1][2].axis("off")
legend_handles = [Patch(facecolor=STRATEGY_COLORS[s], label=f"{s} — {STRATEGY_NAMES[s]}") for s in strategies]
axes[1][2].legend(handles=legend_handles, loc="center", frameon=True, fontsize=12)

plt.suptitle("SonarQube — Metrics by Strategy (Lower = Better)",
             fontsize=16, fontweight="bold", y=1.01)
fig.tight_layout(h_pad=2.0, w_pad=1.5)
fig.savefig(REPORTS_DIR / "chart_sonarqube_by_strategy.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 2160x1200 with 6 Axes>

---
## Chart 6: SonarQube Issues Heatmap by Version

Security open issues, reliability, maintainability, duplications, and security hotspots per version — grouped by methodology.

In [10]:
heatmap_versions = list(version_order)  # 18 versions in methodology-grouped order
sonar_idx = sonar.set_index("Version")
heatmap_meta = sonar_idx.loc[heatmap_versions, ["Feature", "Strategy"]]

hm_metrics = ["Security", "Hotspots", "Reliability", "Maintainability", "Duplications"]
hm_labels = {
    "Security": "Security (Open)",
    "Hotspots": "Security Hotspots",
    "Reliability": "Reliability (Open)",
    "Maintainability": "Maintainability (Open)",
    "Duplications": "Duplications",
}
hm_data = sonar_idx.loc[heatmap_versions, hm_metrics].T.astype(float)
hm_data.index = [hm_labels[m] for m in hm_metrics]

# Custom annotation: integers for issue/hotspot rows, "x.x%" for Duplications
annot = hm_data.copy().astype(object)
for v in heatmap_versions:
    for row in ["Security (Open)", "Security Hotspots", "Reliability (Open)", "Maintainability (Open)"]:
        annot.loc[row, v] = str(int(hm_data.loc[row, v]))
    dup_val = hm_data.loc["Duplications", v]
    annot.loc["Duplications", v] = f"{dup_val:.1f}%" if dup_val > 0 else "0"

strategy_spans = contiguous_spans(heatmap_meta["Strategy"].tolist())
feature_spans = contiguous_spans(list(zip(heatmap_meta["Strategy"], heatmap_meta["Feature"])))
strategy_colors = {"BP": "#e74c3c", "CE": "#3498db", "SD": "#2ecc71"}

fig, ax = plt.subplots(figsize=(20.5, 6.35))
sns.heatmap(hm_data, annot=annot, fmt="", cmap="YlOrRd",
            linewidths=1, linecolor="white", ax=ax,
            cbar_kws={"label": "Open Issues / Hotspots / %"},
            annot_kws={"fontsize": 10, "fontweight": "bold"})

# Feature dividers sit on cell boundaries; methodology dividers are emphasized.
strategy_boundaries = {end for _, end, _ in strategy_spans[:-1]}
for start, _, _ in feature_spans[1:]:
    if start not in strategy_boundaries:
        ax.axvline(start, color="black", linewidth=1.6, linestyle="--", alpha=0.5)

for start, _, _ in strategy_spans[1:]:
    ax.axvline(start, color="#495057", linewidth=2.5, linestyle="-", alpha=0.6)

# Feature sub-labels above each method block; full methodology labels sit below.
for start, end, (_, feature) in feature_spans:
    ax.text((start + end) / 2, 1.025, feature_short_names.get(feature, feature),
            ha="center", va="bottom", fontsize=9, color="#495057",
            transform=ax.get_xaxis_transform(), clip_on=False)

for start, end, strategy in strategy_spans:
    ax.text((start + end) / 2, -0.58, strategy_names[strategy],
            ha="center", va="top", fontsize=12, fontweight="bold", color=strategy_colors[strategy],
            transform=ax.get_xaxis_transform(), clip_on=False)

ax.set_title("SonarQube — Security Open / Hotspots / Reliability / Maintainability / Duplications by Version",
             fontsize=16, fontweight="bold", pad=24)
ax.set_xlabel("")
ax.set_ylabel("")
ax.tick_params(axis="x", pad=2)
plt.xticks(rotation=45, ha="right", fontsize=10)
plt.yticks(rotation=0, fontsize=12)
fig.subplots_adjust(left=0.075, right=0.965, top=0.80, bottom=0.32)
fig.savefig(REPORTS_DIR / "chart_sonarqube_heatmap.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 3075x952.5 with 2 Axes>

---
## Chart 7: SonarQube Code Duplications by Version

In [11]:
fig, ax = plt.subplots(figsize=(13.5, 5.4))

xv = np.arange(len(sonar))
bar_colors = [STRATEGY_COLORS[s] for s in sonar["Strategy"]]
bars = ax.bar(xv, sonar["Duplications"], color=bar_colors, edgecolor="white", width=0.55)

for bar, val in zip(bars, sonar["Duplications"]):
    if val > 0:
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.4,
                f"{val:.1f}%", ha="center", va="bottom", fontweight="bold", fontsize=9)

draw_methodology_groups(ax, sonar, y=-0.37)

ax.set_title("SonarQube — Code Duplications by Version", fontsize=18, fontweight="bold", pad=14)
ax.set_ylabel("Duplication (%)")
ax.set_xticks(xv)
ax.set_xticklabels(sonar["Version"].astype(str), rotation=45, ha="right")
ax.set_ylim(0, sonar["Duplications"].max() * 1.25 + 2)
ax.grid(axis="y", color=GRID_COLOR)

from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=STRATEGY_COLORS[s], label=f"{s} — {STRATEGY_NAMES[s]}")
                   for s in ["BP", "CE", "SD"]]
ax.legend(handles=legend_elements, loc="upper right", frameon=True, fontsize=11)

fig.subplots_adjust(bottom=0.32)
fig.savefig(REPORTS_DIR / "chart_sonarqube_duplications.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 2025x810 with 1 Axes>

---
## CodeQL Data

Source: GitHub Code Scanning API — open alerts, classified by `security_severity_level` (`high` / `medium`).
Re-verified 2026-06-06 against run [#26876283671](https://github.com/PhatchareePuangjai/chatGPT5/actions/runs/26876283671).

In [12]:
HIGH_COLOR = "#e74c3c"
MED_COLOR  = "#f39c12"

codeql = pd.DataFrame([
    {"Version": "IMBP01", "Feature": "Inventory Management",   "Strategy": "BP", "High": 5, "Medium": 0},
    {"Version": "IMBP02", "Feature": "Inventory Management",   "Strategy": "BP", "High": 0, "Medium": 0},
    {"Version": "IMCE01", "Feature": "Inventory Management",   "Strategy": "CE", "High": 3, "Medium": 1},
    {"Version": "IMCE02", "Feature": "Inventory Management",   "Strategy": "CE", "High": 6, "Medium": 0},
    {"Version": "IMSD01", "Feature": "Inventory Management",   "Strategy": "SD", "High": 1, "Medium": 0},
    {"Version": "IMSD02", "Feature": "Inventory Management",   "Strategy": "SD", "High": 0, "Medium": 0},
    {"Version": "SCBP01", "Feature": "Shopping Cart",          "Strategy": "BP", "High": 0, "Medium": 0},
    {"Version": "SCBP02", "Feature": "Shopping Cart",          "Strategy": "BP", "High": 0, "Medium": 0},
    {"Version": "SCCE01", "Feature": "Shopping Cart",          "Strategy": "CE", "High": 6, "Medium": 0},
    {"Version": "SCCE02", "Feature": "Shopping Cart",          "Strategy": "CE", "High": 1, "Medium": 0},
    {"Version": "SCSD01", "Feature": "Shopping Cart",          "Strategy": "SD", "High": 0, "Medium": 0},
    {"Version": "SCSD02", "Feature": "Shopping Cart",          "Strategy": "SD", "High": 0, "Medium": 0},
    {"Version": "PDBP01", "Feature": "Promotions & Discounts", "Strategy": "BP", "High": 0, "Medium": 0},
    {"Version": "PDBP02", "Feature": "Promotions & Discounts", "Strategy": "BP", "High": 1, "Medium": 0},
    {"Version": "PDCE01", "Feature": "Promotions & Discounts", "Strategy": "CE", "High": 0, "Medium": 1},
    {"Version": "PDCE02", "Feature": "Promotions & Discounts", "Strategy": "CE", "High": 1, "Medium": 0},
    {"Version": "PDSD01", "Feature": "Promotions & Discounts", "Strategy": "SD", "High": 0, "Medium": 2},
    {"Version": "PDSD02", "Feature": "Promotions & Discounts", "Strategy": "SD", "High": 0, "Medium": 0},
])

codeql["Total"]    = codeql["High"] + codeql["Medium"]
codeql["Strategy"] = pd.Categorical(codeql["Strategy"], categories=strategy_order, ordered=True)
codeql["Version"]  = pd.Categorical(codeql["Version"],  categories=version_order,  ordered=True)
codeql = codeql.sort_values("Version").reset_index(drop=True)

codeql_strategy = codeql.groupby("Strategy", observed=False)[["High", "Medium", "Total"]].sum().reindex(strategy_order)
codeql_strategy

,High,Medium,Total
Strategy,,,
BP,6,0,6
CE,17,2,19
SD,1,2,3


---
## Chart 8: CodeQL Security Alerts by Strategy

Stacked High + Medium alerts per strategy — lower is better.

In [13]:
fig, ax = plt.subplots(figsize=(7.8, 5.6))

sx = np.arange(3)
ax.bar(sx, codeql_strategy["High"],   color=HIGH_COLOR, label="High (security_severity_level=high)",   edgecolor="white")
ax.bar(sx, codeql_strategy["Medium"], bottom=codeql_strategy["High"], color=MED_COLOR, label="Medium (security_severity_level=medium)", edgecolor="white")

for i, (h, m) in enumerate(zip(codeql_strategy["High"], codeql_strategy["Medium"])):
    total = int(h + m)
    if h >= 2:
        ax.text(i, h / 2, str(int(h)), ha="center", va="center", fontweight="bold", fontsize=14, color="white")
    if m >= 1:
        ax.text(i, h + m / 2, str(int(m)), ha="center", va="center", fontweight="bold", fontsize=13, color="white")
    if total > 0:
        ax.text(i, total + 0.4, str(total), ha="center", va="bottom", fontweight="bold", fontsize=13)
    else:
        ax.text(i, 0.5, "0", ha="center", va="bottom", fontweight="bold", fontsize=13, color="#2f9e44")

ax.set_title("CodeQL (SAST) — Security Alerts by Strategy", fontsize=18, fontweight="bold", pad=14)
ax.set_ylabel("Number of Alerts")
ax.set_xticks(sx)
ax.set_xticklabels(["BP\n(Basic Prompting)", "CE\n(Context Engineering)", "SD\n(Spec-Driven Dev)"])
ax.set_ylim(0, codeql_strategy["Total"].max() + 4)
ax.grid(axis="y", color=GRID_COLOR)
ax.legend(loc="upper right", frameon=True, fontsize=11)

fig.tight_layout()
fig.savefig(REPORTS_DIR / "chart_codeql_by_strategy.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 1170x840 with 1 Axes>

---
## Chart 9: CodeQL Security Alerts by Version

Stacked High + Medium alerts per version — lower is better.

In [14]:
fig, ax = plt.subplots(figsize=(13.5, 5.8))

xv = np.arange(len(codeql))
ax.bar(xv, codeql["High"],   color=HIGH_COLOR, label="High", edgecolor="white")
ax.bar(xv, codeql["Medium"], bottom=codeql["High"], color=MED_COLOR, label="Medium", edgecolor="white")

for idx, row in codeql.iterrows():
    high = int(row["High"])
    medium = int(row["Medium"])
    total = int(row["Total"])

    if high > 0:
        ax.text(idx, high / 2, str(high), ha="center", va="center",
                fontsize=9, fontweight="bold", color="white")
    if medium > 0:
        ax.text(idx, high + medium / 2, str(medium), ha="center", va="center",
                fontsize=9, fontweight="bold", color="white")
    if total > 1 and high > 0 and medium > 0:
        ax.text(idx, total + 0.2, str(total), ha="center", va="bottom",
                fontsize=8.5, fontweight="bold", color="#212529")

draw_methodology_groups(ax, codeql, y=-0.37)

ax.set_title("CodeQL (SAST) — Security Alerts by Version", fontsize=18, fontweight="bold", pad=14)
ax.set_ylabel("Number of Alerts")
ax.set_xticks(xv)
ax.set_xticklabels(codeql["Version"].astype(str), rotation=45, ha="right")
ax.set_ylim(0, codeql["Total"].max() + 3)
ax.grid(axis="y", color=GRID_COLOR)
ax.legend(loc="upper right", frameon=True, fontsize=11)

fig.subplots_adjust(bottom=0.32)
fig.savefig(REPORTS_DIR / "chart_codeql_by_version.png", bbox_inches="tight")
display(fig)
plt.close(fig)

<Figure size 2025x870 with 1 Axes>